In [1]:
import pandas as pd

file_path = r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\Fatality rate\fatality rate_variables_GTD.xlsx"

# Read both sheets
nkill_df = pd.read_excel(file_path, sheet_name="nkill")
nwound_df = pd.read_excel(file_path, sheet_name="nwound")

# Convert nkill
nkill_long = nkill_df.melt(
    id_vars=["Country"], 
    var_name="Year", 
    value_name="nkill"
)

# Convert nwound
nwound_long = nwound_df.melt(
    id_vars=["Country"], 
    var_name="Year", 
    value_name="nwound"
)

panel_df = pd.merge(
    nkill_long,
    nwound_long,
    on=["Country", "Year"],
    how="inner"
)

panel_df["Year"] = panel_df["Year"].astype(int)
panel_df = panel_df.sort_values(by=["Country", "Year"])

# Avoid division by zero
panel_df["fatality_rate"] = panel_df["nkill"] / (panel_df["nkill"] + panel_df["nwound"])

# Handle cases where nkill + nwound = 0
panel_df["fatality_rate"] = panel_df["fatality_rate"].fillna(0)

panel_df

,Country,Year,nkill,nwound,fatality_rate
0,Afghanistan,1970,NaN,NaN,0.000000
204,Afghanistan,1971,NaN,NaN,0.000000
408,Afghanistan,1972,NaN,NaN,0.000000
612,Afghanistan,1973,0.0,1.0,0.000000
816,Afghanistan,1974,NaN,NaN,0.000000
...,...,...,...,...,...
9383,Zimbabwe,2016,NaN,NaN,0.000000
9587,Zimbabwe,2017,0.0,1.0,0.000000
9791,Zimbabwe,2018,2.0,47.0,0.040816
9995,Zimbabwe,2019,0.0,0.0,0.000000


In [2]:
socio_eco=pd.read_excel(r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\Fatality rate\socio-economic_fatality.xlsx")
socio_eco

,Country,Year,Country Code,GDP,Total_Population,Poverty_Headcount,Secondary_Enrollment,Unemployment_Rate,Inequality_Measure
0,Afghanistan,1970,AFG,NaN,11290128.0,NaN,8.36410,NaN,NaN
1,Afghanistan,1971,AFG,NaN,11567667.0,NaN,9.37827,NaN,NaN
2,Afghanistan,1972,AFG,NaN,11853696.0,NaN,10.41424,NaN,NaN
3,Afghanistan,1973,AFG,NaN,12157999.0,NaN,10.93218,NaN,NaN
4,Afghanistan,1974,AFG,NaN,12469127.0,NaN,11.09488,NaN,NaN
...,...,...,...,...,...,...,...,...,...
10540,Zimbabwe,2016,ZWE,0.755794,14600294.0,NaN,NaN,NaN,0.5178
10541,Zimbabwe,2017,ZWE,4.734411,14812482.0,0.446569,NaN,NaN,0.5198
10542,Zimbabwe,2018,ZWE,5.009922,15034452.0,NaN,NaN,NaN,0.5523
10543,Zimbabwe,2019,ZWE,-6.332450,15271368.0,0.492199,NaN,7.373,0.5848


In [3]:
# Ensure same format before merging

# Clean panel_df (terrorism data)
panel_df["Country"] = panel_df["Country"].astype(str).str.strip().str.lower()
panel_df["Year"] = panel_df["Year"].astype(int)

# Clean WGI data
socio_eco["Country"] = socio_eco["Country"].astype(str).str.strip().str.lower()
socio_eco["Year"] = socio_eco["Year"].astype(int)

# Merge
final_df = pd.merge(
    panel_df,
    socio_eco,
    on=["Country", "Year"],
    how="left"   # 🔥 recommended
)

final_df

,Country,Year,nkill,nwound,fatality_rate,Country Code,GDP,Total_Population,Poverty_Headcount,Secondary_Enrollment,Unemployment_Rate,Inequality_Measure
0,afghanistan,1970,NaN,NaN,0.000000,AFG,NaN,11290128.0,NaN,8.36410,NaN,NaN
1,afghanistan,1971,NaN,NaN,0.000000,AFG,NaN,11567667.0,NaN,9.37827,NaN,NaN
2,afghanistan,1972,NaN,NaN,0.000000,AFG,NaN,11853696.0,NaN,10.41424,NaN,NaN
3,afghanistan,1973,0.0,1.0,0.000000,AFG,NaN,12157999.0,NaN,10.93218,NaN,NaN
4,afghanistan,1974,NaN,NaN,0.000000,AFG,NaN,12469127.0,NaN,11.09488,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
10261,zimbabwe,2016,NaN,NaN,0.000000,ZWE,0.755794,14600294.0,NaN,NaN,NaN,0.5178
10262,zimbabwe,2017,0.0,1.0,0.000000,ZWE,4.734411,14812482.0,0.446569,NaN,NaN,0.5198
10263,zimbabwe,2018,2.0,47.0,0.040816,ZWE,5.009922,15034452.0,NaN,NaN,NaN,0.5523
10264,zimbabwe,2019,0.0,0.0,0.000000,ZWE,-6.332450,15271368.0,0.492199,NaN,7.373,0.5848


In [4]:
final_df.to_excel("fatality rate_socio_wgi.xlsx",index=False)